In [1]:
!pip install paddlepaddle
!pip install paddleocr

In [2]:
pip install ultralytics

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import cv2
import csv 
import easyocr
from ultralytics import YOLO
from paddleocr import PaddleOCR
import re
import numpy as np

Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [4]:
PROJECT_ROOT = "/Users/binhminh/Project-II"
YOLO_WEIGHTS = os.path.join(PROJECT_ROOT, "runs/detect/runs_yolo11/plate_detection/weights/best.pt")
TEST_IMAGE_DIR = os.path.join(PROJECT_ROOT, "yolo_dataset/images/test")
OUTPUT_CSV = os.path.join(PROJECT_ROOT, "OCR/ocr_results_paddle.csv")

print("Loading YOLO11...")
yolo_model = YOLO(YOLO_WEIGHTS)

print("Initializing PaddleOCR...")
ocr_model = PaddleOCR(use_angle_cls=False, lang='en')

Loading YOLO11...
Initializing PaddleOCR...


/var/folders/5b/lg2597jn17z4b_yr082m88tm0000gn/T/ipykernel_72220/2021238857.py:10: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr_model = PaddleOCR(use_angle_cls=False, lang='en')
/opt/anaconda3/lib/python3.13/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/binhminh/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/binhminh/.paddlex/official_m

In [5]:
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
valid_extensions = ('.jpg', '.jpeg', '.png')
image_files = [f for f in os.listdir(TEST_IMAGE_DIR) if f.lower().endswith(valid_extensions)]

print(f"Tổng số ảnh cần xử lý: {len(image_files)}")


Tổng số ảnh cần xử lý: 350


In [6]:
# Xem cấu trúc trả về của OCR
print("Kiểm tra cấu trúc trả về của OCR:")
test_image_path = os.path.join(TEST_IMAGE_DIR, image_files[0])
test_image = cv2.imread(test_image_path)
test_results = ocr_model.ocr(test_image)
print("Cấu trúc trả về của OCR:")
print(test_results)

Kiểm tra cấu trúc trả về của OCR:


/var/folders/5b/lg2597jn17z4b_yr082m88tm0000gn/T/ipykernel_72220/586236839.py:5: DeprecationWarning: Please use `predict` instead.
  test_results = ocr_model.ocr(test_image)


Cấu trúc trả về của OCR:
[{'input_path': None, 'page_index': None, 'doc_preprocessor_res': {'input_path': None, 'page_index': None, 'input_img': array([[[155, ..., 132],
        ...,
        [116, ..., 103]],

       ...,

       [[ 37, ...,  71],
        ...,
        [140, ..., 108]]], shape=(303, 472, 3), dtype=uint8), 'model_settings': {'use_doc_orientation_classify': True, 'use_doc_unwarping': True}, 'angle': 180, 'rot_img': array([[[  0, ...,   0],
        ...,
        [  0, ...,   0]],

       ...,

       [[  0, ...,   0],
        ...,
        [153, ..., 131]]], shape=(303, 472, 3), dtype=uint8), 'output_img': array([[[134, ..., 128],
        ...,
        [ 94, ...,  96]],

       ...,

       [[120, ..., 108],
        ...,
        [ 95, ..., 110]]], shape=(303, 472, 3), dtype=uint8)}, 'dt_polys': [array([[205,  98],
       ...,
       [205, 132]], shape=(4, 2), dtype=int16), array([[214, 126],
       ...,
       [213, 162]], shape=(4, 2), dtype=int16)], 'model_settings': {'use_

In [7]:
def sort_multiline_text(texts, dt_polys):
    """
    Đầu vào: List text và list boxes từ PaddleOCR.
    Đầu ra: List text đã được sắp xếp từ trên xuống dưới.
    """
    # Sửa lỗi: Kiểm tra độ dài thay vì dùng 'if not boxes' trực tiếp trên numpy array
    if not texts or not dt_polys or len(texts) == 0: return []
    
    combined = []
    for i in range(len(texts)):
        try:
            # dt_polys là mảng numpy chứa 4 điểm [[x,y], [x,y], [x,y], [x,y]]
            # Lấy y nhỏ nhất của bounding box đó
            y_min = np.min(dt_polys[i][:, 1])
        except Exception:
            y_min = 0
            
        combined.append({"text": str(texts[i]).upper(), "y": y_min})
    
    # Sắp xếp từ trên xuống dưới
    sorted_data = sorted(combined, key=lambda x: x["y"])
    return [item["text"] for item in sorted_data]

In [8]:
def enhance_plate_image(plate_crop):
    """Chỉ dùng Resize và CLAHE để làm rõ nét, tuyệt đối không dùng WarpPerspective vì YOLO đã cắt sát"""
    # Phóng to ảnh gấp 2 lần giúp OCR đọc nét đứt tốt hơn
    res_img = cv2.resize(plate_crop, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    gray_res = cv2.cvtColor(res_img, cv2.COLOR_BGR2GRAY)
    
    # Cân bằng sáng cục bộ (CLAHE)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    contrasted = clahe.apply(gray_res)
    
    return cv2.cvtColor(contrasted, cv2.COLOR_GRAY2BGR)

In [9]:
import re

def postprocess_plate_text(lines):
    if not lines: return "NO_TEXT", False
    # Làm sạch ký tự lạ
    cleaned = [re.sub(r'[^A-Z0-9]', '', str(l)).upper() for l in lines if l]
    if not cleaned: return "NO_TEXT", False

    # Định nghĩa các bảng Map để dùng chung
    flip_map = {'9':'6', '6':'9', 'L':'1', 'I':'1', 'A':'V', 'V':'A', 'E':'3', '3':'E', 'S':'5', '5':'S', '7':'2', '2':'7', 'J':'C', 'C':'J', 'Z':'2'}
    char_to_num = {'E':'6', 'G':'6', 'S':'5', 'B':'8', 'O':'0', 'D':'0', 'Z':'2', 'L':'1', 'I':'1', 'T':'7', 'W':'7', 'A':'4'}
    num_to_char = {'6':'G', '5':'S', '8':'B', '0':'D', '2':'Z', '1':'L', '7':'T', '4':'A', '3':'E'}

    is_flipped = False
    l1_raw = cleaned[0]
    l2_raw = cleaned[1] if len(cleaned) >= 2 else ""

    # --- BƯỚC 1: XỬ LÝ LẬT NGƯỢC (ĐÃ SỬA: Kiểm tra chéo 2 dòng) ---
    # Nếu dòng 1 bắt đầu bằng chữ (E, S, G...) hoặc dòng 2 trông giống mã tỉnh hơn dòng 1
    if (len(l1_raw) >= 1 and l1_raw[0].isalpha()) or (len(l2_raw) >= 2 and l2_raw[:2].isdigit() and not l1_raw[:2].isdigit()):
        is_flipped = True
        # Khi lật 180 độ: Dòng 2 thực tế là dòng 1 bị lộn, Dòng 1 thực tế là dòng 2 bị lộn
        # Ta đảo ngược thứ tự chuỗi và xoay ký tự
        line1 = "".join([flip_map.get(c, c) for c in l2_raw[::-1]]) if l2_raw else l1_raw
        line2 = "".join([flip_map.get(c, c) for c in l1_raw[::-1]])
    else:
        line1 = l1_raw
        line2 = l2_raw

    # --- BƯỚC 2: ÉP LUẬT DÒNG 1 (XY-ZT) (ĐÃ SỬA: X từ 1-9, Z là chữ) ---
    l1 = list(line1)
    
    # Vị trí 0 (X): Phải là số từ 1-9
    if len(l1) >= 1:
        if not l1[0].isdigit() or l1[0] == '0':
            l1[0] = char_to_num.get(l1[0], l1[0])
            if l1[0] == '0': l1[0] = '8' # Ví dụ: nhầm 8 thành 0 thì sửa lại thành 8 (mã tỉnh không bắt đầu bằng 0)

    # Vị trí 1 (Y): Phải là số 0-9
    if len(l1) >= 2:
        if not l1[1].isdigit():
            l1[1] = char_to_num.get(l1[1], l1[1])

    # Vị trí 2 (Z): Bắt buộc là CHỮ
    if len(l1) >= 3:
        if l1[2].isdigit():
            # Ưu tiên map 5->S, 3->S/E, 1->L...
            l1[2] = num_to_char.get(l1[2], l1[2])
        elif l1[2] not in "ABCDEFGHIJKLMNOPQRSTUVWXYZ": # Nếu là ký tự rác
            l1[2] = 'S' # Default về S nếu không nhận diện được

    # Vị trí 3 (T): Chữ hoặc Số (Thường là số ở xe máy phổ thông)
    # Không cần ép chặt nhưng nếu là ký tự dễ nhầm thì ưu tiên số
    if len(l1) >= 4:
        if l1[3] in ['S', 'E', 'G', 'B']: # Các chữ hay nhầm với số 5, 3, 6, 8
            l1[3] = char_to_num.get(l1[3], l1[3])

    l1_res = "".join(l1)
    if len(l1_res) > 2: l1_res = f"{l1_res[:2]}-{l1_res[2:]}"

    # --- BƯỚC 3: ÉP LUẬT DÒNG 2 (Số thứ tự) (ĐÃ SỬA: ÉP 100% SỐ) ---
    l2_res = "".join([char_to_num.get(c, c) if not c.isdigit() else c for c in line2])
    if len(l2_res) == 5: 
        l2_res = f"{l2_res[:3]}.{l2_res[3:]}"

    return f"{l1_res} {l2_res}".strip(), is_flipped

In [11]:
import pandas as pd
import os
import cv2

suspicious_images = [] # Danh sách lưu ảnh bị ngược để debug
results_list = []

print("Bắt đầu quá trình OCR nâng cao (Tích hợp Flip-Correction)...")

for img_file in image_files:
    img_path = os.path.join(TEST_IMAGE_DIR, img_file)
    img = cv2.imread(img_path)
    if img is None: continue
        
    # 1. YOLO Detection
    yolo_results = yolo_model(img, verbose=False)
    boxes = yolo_results[0].boxes.xyxy.cpu().numpy()
    
    if len(boxes) == 0:
        results_list.append({"Filename": img_file, "Predicted_Plate": "NO_PLATE", "Confidence": 0.0, "Is_Flipped": False})
        continue

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)
        padding = 5
        plate_crop = img[max(0, y1-padding):min(img.shape[0], y2+padding), 
                         max(0, x1-padding):min(img.shape[1], x2+padding)]
    
        processed_crop = enhance_plate_image(plate_crop)

        ocr_res = ocr_model.predict(processed_crop)
        
        full_text = "OCR_FAILED"
        avg_conf = 0.0
        was_flipped = False
        
        if ocr_res and len(ocr_res) > 0 and ocr_res[0] is not None:
            data = ocr_res[0]
            if 'rec_texts' in data and len(data['rec_texts']) > 0:
                raw_texts = data['rec_texts']
                raw_polys = data.get('dt_polys', [])
                
                # BƯỚC A: Sắp xếp dòng theo tọa độ Y
                sorted_lines = sort_multiline_text(raw_texts, raw_polys)
                
                # BƯỚC B: TÍCH HỢP KIỂM TRA ĐẢO NGƯỢC (FLIP-FIX)
                full_text, was_flipped = postprocess_plate_text(sorted_lines)
                
                if was_flipped:
                    suspicious_images.append({"file": img_file, "raw": sorted_lines, "fixed": full_text})
                    print(f"[⚠️ FIXED FLIP] {img_file}: {sorted_lines} -> {full_text}")
                
                # Tính độ tự tin
                scores = data.get('rec_scores', [0.0] * len(raw_texts))
                avg_conf = sum(scores) / len(scores)
                
                if not was_flipped:
                    print(f"[OK] {img_file} -> Final: {full_text} (Conf: {avg_conf:.2f})")
            else:
                full_text = "TEXT_NOT_FOUND"

        # Lưu kết quả
        results_list.append({
            "Filename": img_file, 
            "Predicted_Plate": full_text, 
            "Confidence": round(float(avg_conf), 4),
            "Is_Flipped": was_flipped # Đánh dấu để bạn dễ lọc trong CSV
        })

# 3. Lưu kết quả ra CSV
df = pd.DataFrame(results_list)
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f"\n Hoàn thành! Đã sửa {len(suspicious_images)} ảnh bị ngược.")
print(f"File kết quả: {OUTPUT_CSV}")

🚀 Bắt đầu quá trình OCR nâng cao (Tích hợp Flip-Correction)...
[OK] 0006_06035_b.jpg -> Final: 59-U1 396.19 (Conf: 0.99)
[OK] 0143_04789_b.jpg -> Final: 59-M1 563.78 (Conf: 0.93)
[OK] 0249_06036_b.jpg -> Final: 67-G1 067.16 (Conf: 0.98)
[OK] 0249_00316_b.jpg -> Final: 59-U1 497.26 (Conf: 0.99)
[OK] 0228_01938_b.jpg -> Final: 59-N1 014.25 (Conf: 0.99)
[OK] 0222_08157_b.jpg -> Final: 62-S3 5031 (Conf: 0.98)
[OK] 0246_04669_b.jpg -> Final: 60-C2 006.84 (Conf: 0.99)
[OK] 0322_05697_b.jpg -> Final: 59-U1 528.84 (Conf: 0.99)
[OK] 0442_01393_b.jpg -> Final: 62-K1 095.71 (Conf: 0.99)
[OK] 0139_07352_b.jpg -> Final: 59-F1 238.65 (Conf: 0.95)
[OK] 0251_04333_b.jpg -> Final: 54-S8 5452 (Conf: 0.99)
[OK] 0326_04993_b.jpg -> Final: 77-F1 253.89 (Conf: 1.00)
[OK] 0221_08150_b.jpg -> Final: 16-M9 0826 (Conf: 1.00)
[OK] 0131_00733_b.jpg -> Final: 59-Y1 608.11 (Conf: 1.00)
[OK] 0403_08185_b.jpg -> Final: 66-N1 105.47 (Conf: 0.98)
[OK] 0153_00460_b.jpg -> Final: 59-T1 599.73 (Conf: 0.99)
[OK] 0250_06810

In [33]:
# Debug flipped image
if suspicious_images:
    print("\nDanh sách ảnh nghi ngờ bị ngược đã được sửa:")
    for item in suspicious_images:
        print(f"- {item['file']}: Raw: {item['raw']} -> Fixed: {item['fixed']}")


Danh sách ảnh nghi ngờ bị ngược đã được sửa:
- 0139_05267_b.jpg: Raw: ['L0900', 'V3-0L'] -> Fixed: 10-EA 006.01
- 0511_02302_b.jpg: Raw: ['L7LL9', 'LA-63'] -> Fixed: 69-V1 611.21
- 0125_02242_b.jpg: Raw: ['5Y-V2', '158.00'] -> Fixed: 80-B51 74Y5
- 0150_06837_b.jpg: Raw: ['L7-R1', '4301'] -> Fixed: 10-E4 1R21
- 0242_05960_b.jpg: Raw: ['5Q-K2', '012.24'] -> Fixed: 47-T10 7KQ5
- 0440_08382_b.jpg: Raw: ['S1-D1', '478.34'] -> Fixed: 46-B24 1015
- 0416_05808_b.jpg: Raw: ['L11L7', '53-69'] -> Fixed: 69-E5 211.11


In [35]:
import pandas as pd
import re

def check_line1_format(plate_str):
    """
    Kiểm tra định dạng dòng 1 biển xe máy VN: XY-ZT
    X: 1-9
    Y: 0-9
    Z: Chữ cái (A-Z)
    T: Chữ hoặc Số (A-Z0-9)
    """
    if pd.isna(plate_str) or plate_str == "NO_PLATE" or " " not in plate_str:
        return False
    
    # Tách lấy dòng 1 (phần trước dấu cách)
    line1 = plate_str.split(' ')[0]
    # Loại bỏ dấu gạch ngang
    clean_l1 = line1.replace('-', '')
    pattern = r'^[1-9][0-9][A-Z][A-Z0-9]$'
    
    return bool(re.match(pattern, clean_l1))

# 1. Đọc file CSV kết quả
df = pd.read_csv(OUTPUT_CSV)

# 2. Phân loại
df['Is_Line1_Valid'] = df['Predicted_Plate'].apply(check_line1_format)

# 3. Lọc ra các biển bị sai định dạng dòng 1
invalid_df = df[df['Is_Line1_Valid'] == False]

# 4. Thống kê và Hiển thị
total = len(df)
invalid_count = len(invalid_df)

print(f"--- BÁO CÁO DEBUG BIỂN SỐ ---")
print(f"Tổng số biển đã xử lý: {total}")
print(f"Số biển dòng 1 SAI định dạng: {invalid_count} ({invalid_count/total*100:.2f}%)")
print(f"-----------------------------\n")

if invalid_count > 0:
    print("Danh sách các biển nghi vấn (Dòng 1 không phải XX-YX):")
    # Hiển thị Filename và Predicted_Plate để bạn check ảnh gốc
    print(invalid_df[['Filename', 'Predicted_Plate', 'Confidence']].to_string(index=False))
else:
    print(" Tuyệt vời! Tất cả các dòng 1 đều đúng định dạng.")

--- BÁO CÁO DEBUG BIỂN SỐ ---
Tổng số biển đã xử lý: 350
Số biển dòng 1 SAI định dạng: 7 (2.00%)
-----------------------------

Danh sách các biển nghi vấn (Dòng 1 không phải XX-YX):
        Filename Predicted_Plate  Confidence
0154_02257_b.jpg     11-G20 1019      0.7298
0125_02242_b.jpg     80-B51 74Y5      0.8384
0256_05284_b.jpg      82-91 6579      0.6676
0519_02301_b.jpg     57-T89 5469      0.6458
0440_08382_b.jpg     46-B24 1015      0.9667
0317_08220_b.jpg     79-B60 2N62      0.7508
0502_05172_b.jpg     86-Z29 1369      0.6455


In [39]:
import os
import cv2
import pandas as pd
import numpy as np
from tqdm import tqdm

def deskew_plate(img, points):
    pts = np.array(points, dtype="float32")
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]

    (tl, tr, br, bl) = rect
    widthA = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
    widthB = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))
    maxWidth = max(int(widthA), int(widthB))

    heightA = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
    heightB = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))
    maxHeight = max(int(heightA), int(heightB))

    dst = np.array([[0, 0], [maxWidth - 1, 0], [maxWidth - 1, maxHeight - 1], [0, maxHeight - 1]], dtype="float32")
    M = cv2.getPerspectiveTransform(rect, dst)
    return cv2.warpPerspective(img, M, (maxWidth, maxHeight))

def smart_sort_plate_lines(texts):
    line1, line2 = "", ""
    for t in texts:
        t_clean = t.replace(" ", "").replace("-", "").replace(".", "")
        if any(c.isalpha() for c in t_clean): line1 = t
        elif any(c.isdigit() for c in t_clean): line2 = t
    return [line1, line2] if line1 and line2 else texts


THRESHOLD = 0.90
df = pd.read_csv(OUTPUT_CSV)

# Lấy các file sai định dạng
invalid_files = df[df['Is_Line1_Valid'] == False]['Filename'].tolist()

# Lấy các file có Conf < Threshold
low_conf_files = df[df['Confidence'] < THRESHOLD]['Filename'].tolist()

# Gộp 2 danh sách và loại bỏ trùng lặp bằng set()
target_files = list(set(invalid_files + low_conf_files))

print(f" --- THỐNG KÊ MẪU KHÓ ---")
print(f"   - Số mẫu sai định dạng: {len(invalid_files)}")
print(f"   - Số mẫu Conf < {THRESHOLD}: {len(low_conf_files)}")
print(f" Tổng số ảnh độc lập cần OCR nhìn lại: {len(target_files)}")
print("-" * 40)

# VÒNG LẶP CHO OCR NHÌN LẠI (Tích hợp Deskew & Flip)
fixed_data = []

for filename in tqdm(target_files):
    img_path = os.path.join(TEST_IMAGE_DIR, filename)
    img_original = cv2.imread(img_path)
    if img_original is None: continue

    # Lấy thông tin cũ để so sánh
    old_row = df[df['Filename'] == filename].iloc[0]
    old_conf = old_row['Confidence']
    old_valid = old_row['Is_Line1_Valid']

    res = ocr_model.predict(img_original)
    
    if res and len(res[0]['dt_polys']) > 0:
        # Làm phẳng ảnh
        all_boxes = np.array(res[0]['dt_polys']) 
        pts = all_boxes.reshape(-1, 2).astype(np.float32)
        rect = cv2.minAreaRect(pts)
        box = cv2.boxPoints(rect).astype(np.float32)
        img_warped = deskew_plate(img_original, box)
        
        # Tạo 2 phiên bản để check: Phẳng Thuận và Phẳng Ngược
        img_warped_flip = cv2.rotate(img_warped, cv2.ROTATE_180)
        
        best_new_plate = None
        best_new_conf = 0.0

        # Thử nghiệm trên cả 2 ảnh
        for img_version in [img_warped, img_warped_flip]:
            processed = enhance_plate_image(img_version)
            res_version = ocr_model.predict(processed)
            
            if res_version and len(res_version[0]['rec_texts']) >= 2:
                raw_texts = res_version[0]['rec_texts']
                sorted_lines = smart_sort_plate_lines(raw_texts)
                full_text, _ = postprocess_plate_text(sorted_lines)
                
                if check_line1_format(full_text):
                    avg_conf = np.mean(res_version[0]['rec_scores'])
                    # Lưu lại nếu phiên bản này có Conf cao nhất
                    if avg_conf > best_new_conf:
                        best_new_conf = avg_conf
                        best_new_plate = full_text
        
        # --- ĐIỀU KIỆN CẬP NHẬT ---
        # Chỉ cập nhật nếu: Tìm ra biển hợp lệ VÀ (biển cũ bị sai định dạng HOẶC Conf mới cao hơn Conf cũ)
        if best_new_plate:
            if (not old_valid) or (best_new_conf > old_conf):
                fixed_data.append({
                    "Filename": filename,
                    "New_Plate": best_new_plate,
                    "New_Conf": round(float(best_new_conf), 4)
                })

# CẬP NHẬT DATAFRAME & XUẤT BÁO CÁO
for item in fixed_data:
    mask = df['Filename'] == item['Filename']
    df.loc[mask, 'Predicted_Plate'] = item['New_Plate']
    df.loc[mask, 'Confidence'] = item['New_Conf']
    df.loc[mask, 'Is_Line1_Valid'] = True

print(f"\n KẾT QUẢ:")
print(f"   - Đã cải thiện/sửa lỗi thành công: {len(fixed_data)} / {len(target_files)} biển")
print(f"   - Tỷ lệ thành công của module: {(len(fixed_data)/len(target_files))*100:.2f}%")

df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f" Đã cập nhật kết quả mới nhất vào: {OUTPUT_CSV}")

📊 --- THỐNG KÊ MẪU KHÓ ---
   - Số mẫu sai định dạng: 4
   - Số mẫu Conf < 0.9: 15
🔍 Tổng số ảnh độc lập cần OCR nhìn lại: 16
----------------------------------------


100%|██████████| 16/16 [00:25<00:00,  1.57s/it]


✅ KẾT QUẢ CỨU VÃN:
   - Đã cải thiện/sửa lỗi thành công: 0 / 16 biển
   - Tỷ lệ thành công của module: 0.00%
📝 Đã cập nhật kết quả mới nhất vào: /Users/binhminh/Project-II/OCR/ocr_results_paddle.csv


In [40]:
# Hiển thị danh sách các biển đã được sửa
if fixed_data:
    print("\nDanh sách các biển đã được sửa:")
    for item in fixed_data:
        print(f"- {item['Filename']}: New Plate: {item['New_Plate']} (Conf: {item['New_Conf']})")